# SQL 7: types, functions and JSON

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-07-types-functions-and-json.ipynb)

A query is only as correct as the types it works with. This chapter covers the data types you will meet,
how Spark converts between them, the string and date functions used every day, and how to work with arrays,
structs and JSON. Each section ends with the mistake it is most likely to prevent.

It ends with practice questions and answers.

## Setup

Install PySpark, start a Spark session, and create the example tables.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyspark": "pyspark>=4.0,<4.1"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW customers AS
SELECT * FROM VALUES
  (1, 'Acme Corp',    'North', DATE '2022-01-05'),
  (2, 'Beta LLC',     'North', DATE '2022-02-10'),
  (3, 'Cyan Inc',     'South', DATE '2022-03-15'),
  (4, 'Delta Co',     'South', DATE '2022-04-20'),
  (5, 'Echo Ltd',     'West',  DATE '2022-05-25'),
  (6, 'Foxtrot GmbH', 'West',  DATE '2022-06-30'),
  (7, 'Gamma SA',     'East',  DATE '2023-01-10'),
  (8, 'Hotel BV',     'East',  DATE '2023-02-15')
AS customers(customer_id, name, region, signup_date)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW orders AS
SELECT * FROM VALUES
  (1,  1, 'completed', 100, DATE '2023-01-05'),
  (2,  1, 'completed', 200, DATE '2023-02-10'),
  (3,  1, 'completed', 150, DATE '2023-03-15'),
  (4,  2, 'completed', 300, DATE '2023-01-20'),
  (5,  2, 'cancelled', 250, DATE '2023-02-05'),
  (6,  2, 'completed', 400, DATE '2023-03-20'),
  (7,  3, 'completed', 120, DATE '2023-01-25'),
  (8,  3, 'completed', 180, DATE '2023-02-28'),
  (9,  4, 'pending',   220, DATE '2023-02-14'),
  (10, 4, 'completed', 320, DATE '2023-03-30'),
  (11, 5, 'completed', 500, DATE '2023-01-02'),
  (12, 5, 'completed', 450, DATE '2023-03-05'),
  (13, 6, 'completed', 275, DATE '2023-02-18'),
  (14, 6, 'cancelled', 325, DATE '2023-03-25')
AS orders(order_id, customer_id, status, amount, order_date)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW contacts AS
SELECT * FROM VALUES (1, 'ana@x.com'), (2, 'ben@y.org'), (3, 'cy@x.com')
AS contacts(id, email)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW shipments AS
SELECT * FROM VALUES
  (1, '{"city": "Pune", "zip": "411001"}'),
  (2, '{"city": "Delhi", "zip": "110001"}')
AS shipments(id, address_json)
""")

Besides `customers` and `orders`, two small tables support this chapter. `contacts` has three email
addresses, two at `x.com` and one at `y.org`. `shipments` stores each address as a JSON **string**, not as a
structured value, which is how API payloads usually land in a warehouse.

In [ ]:
assert spark.sql("SELECT COUNT(*) FROM contacts").collect() == [(3,)]
assert spark.sql("SELECT typeof(address_json) FROM shipments LIMIT 1").collect() == [("string",)]

## 1. Data types

Every column and every expression has a type. The type decides which values are allowed, how they compare and
sort, and what arithmetic does. The common ones are:

| Type | Holds | Example literal |
| --- | --- | --- |
| `INT`, `BIGINT` | whole numbers (32-bit and 64-bit) | `42`, `42L` |
| `DECIMAL(p, s)` | exact numbers with a fixed scale, such as money | `1.5` |
| `DOUBLE` | approximate floating-point numbers | `1.5D` |
| `STRING` | text | `'abc'` |
| `DATE`, `TIMESTAMP` | calendar dates, and dates with a time | `DATE '2023-01-05'` |
| `BOOLEAN` | true or false | `true` |
| `ARRAY`, `STRUCT`, `MAP` | collections and records | `array(1, 2)` |

`typeof` shows the type of an expression. Note that the literal `1.5` is a `DECIMAL`, not a `DOUBLE`.

### Where you'll use it

- **Schema design:** choosing `DECIMAL(12,2)` for money and `TIMESTAMP` for events prevents years of rounding and time-zone bugs.
- **Reading error messages:** most load failures name a type: an overflow, a bad cast, a precision limit.

In [ ]:
q = """
SELECT typeof(42)              AS int_literal,
       typeof(42L)             AS bigint_literal,
       typeof(1.5)             AS decimal_literal,
       typeof(1.5D)            AS double_literal,
       typeof('abc')           AS string_literal,
       typeof(DATE '2023-01-05') AS date_literal
"""
spark.sql(q).show(truncate=False)
assert spark.sql(q).collect() == [("int", "bigint", "decimal(2,1)", "double", "string", "date")]

### Precision and scale

In `DECIMAL(p, s)`, the **precision** `p` is the total number of digits and the **scale** `s` is how many of
them come after the decimal point. `1.5` has two digits, one after the point, so it is `DECIMAL(2,1)`.
`DECIMAL(5,2)` holds at most `999.99`. A value that needs more digits, such as `1234.5`, does not fit: `CAST`
raises an error and `try_cast` returns `NULL`.

The same limit applies to integers. An `INT` holds up to 2,147,483,647. In ANSI mode adding 1 to that raises an
overflow error; `try_add` returns `NULL` instead.

In [ ]:
from pyspark.errors import ArithmeticException

q = "SELECT try_cast(1234.5 AS DECIMAL(5,2)) AS too_wide, CAST(999.99 AS DECIMAL(5,2)) AS fits, try_add(2147483647, 1) AS overflow"
spark.sql(q).show()

from decimal import Decimal

assert spark.sql(q).collect() == [(None, Decimal("999.99"), None)]

raised = False
try:
    spark.sql("SELECT CAST(1234.5 AS DECIMAL(5,2))").collect()
except ArithmeticException as error:
    raised = True
    print(str(error)[:70])
assert raised

### Division

**Division is a common trap.** In Spark, `/` on two integers returns a `DOUBLE`, so `10 / 4` is `2.5`. Use
`DIV` when you want integer division: it drops the fractional part and returns `2`.

Dividing by zero raises an error in ANSI mode. `try_divide(a, b)` returns `NULL` instead, and so does
`a / NULLIF(b, 0)`: `NULLIF(b, 0)` turns a zero divisor into `NULL`, and dividing by `NULL` gives `NULL`.

In [ ]:
q = """
SELECT 10 / 4              AS true_division,
       typeof(10 / 4)      AS division_type,
       10 DIV 4            AS integer_division,
       try_divide(10, 0)   AS try_divide_by_zero,
       10 / NULLIF(0, 0)   AS nullif_divide_by_zero
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(2.5, "double", 2, None, None)]

## 2. Casting and implicit conversion

`CAST(value AS type)` converts a value explicitly. Spark also converts types implicitly when it can, for example
when you add a string to a number. This is called **implicit conversion** or **coercion**. Spark 4 runs in ANSI
mode by default, so a malformed cast raises an error instead of returning NULL. Use `try_cast` when bad input
should become NULL.

### Where you'll use it

- **Ingesting strings:** every CSV load casts text columns to numbers and dates, and a bad cast is the first error most loads hit.
- **Quarantining bad rows:** `WHERE try_cast(col AS INT) IS NULL AND col IS NOT NULL` finds the values that will not convert.

`try_cast('12' AS INT)` is 12 and `try_cast('abc' AS INT)` is `NULL`. In `'10' + 5`, Spark converts the
string `'10'` to a `BIGINT` and adds: the result is 15, of type `bigint`.

In [ ]:
q = """
SELECT try_cast('12'  AS INT) AS valid_text,
       try_cast('abc' AS INT) AS malformed_text
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(12, None)]

q = "SELECT '10' + 5 AS implicit_conversion, typeof('10' + 5) AS result_type"
spark.sql(q).show()
assert spark.sql(q).collect() == [(15, "bigint")]

The same implicit conversion fails when the text is not a number. `'abc' + 5` raises `CAST_INVALID_INPUT` when
the query runs. The cell below catches the error to show it.

In [ ]:
from pyspark.errors import NumberFormatException

raised = False
try:
    spark.sql("SELECT 'abc' + 5").collect()
except NumberFormatException as error:
    raised = True
    print(str(error)[:60])
assert raised

### Casting a decimal to an integer truncates

`CAST(2.7 AS INT)` is `2`, not `3`: the fractional part is dropped, moving toward zero, so `CAST(-2.7 AS INT)`
is `-2`. Use `ROUND` first when you want the nearest whole number.

In [ ]:
q = "SELECT CAST(2.7 AS INT), CAST(-2.7 AS INT), CAST(ROUND(2.7) AS INT)"
spark.sql(q).show()
assert spark.sql(q).collect() == [(2, -2, 3)]

### Dates stored as text

`order_date = '2023-01-05'` works because Spark converts the string to a date. This is a convenience, not a
guarantee. A column of text dates sorts and compares as text, character by character. For dates written as
`dd/MM/yyyy`, `'31/01/2023'` is greater than `'05/02/2023'` because `'3'` comes after `'0'`, so `MAX` picks
31 January over 5 February. Parsing with `to_date` first gives the right answer. Store dates as `DATE` or
`TIMESTAMP`.

In [ ]:
assert spark.sql("SELECT order_id FROM orders WHERE order_date = '2023-01-05'").collect() == [(1,)]

from datetime import date

q = """
SELECT MAX(d) AS max_as_text, MAX(to_date(d, 'dd/MM/yyyy')) AS max_as_date
FROM VALUES ('31/01/2023'), ('05/02/2023') AS t(d)
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("31/01/2023", date(2023, 2, 5))]

## 3. String functions

| Function | Does |
| --- | --- |
| `concat(a, b)`, `concat_ws(sep, a, b)` | join strings; `concat_ws` skips NULLs and puts `sep` between the rest |
| `upper`, `lower`, `trim` | change case; remove spaces at the ends |
| `substring(s, start, length)` | part of a string; `start` is 1-based |
| `length(s)` | number of characters |
| `replace(s, from, to)` | replace every occurrence |
| `split(s, sep)` | split into an array; `sep` is a regular expression |
| `regexp_extract(s, pattern, group)` | the part of `s` matched by a regular expression |
| `LIKE`, `RLIKE` | simple pattern match, or a regular-expression match |

### Where you'll use it

- **Cleaning customer data:** trimming and normalising emails and names before matching records.
- **Parsing codes:** extracting a country code from a product SKU with `SUBSTRING`.

In [ ]:
q = """
SELECT concat_ws(' ', 'Acme', 'Corp')   AS joined,
       upper(trim('  pune  '))           AS cleaned,
       substring('abcdef', 2, 3)         AS middle,
       length('Acme Corp')               AS characters,
       replace('2023-01-05', '-', '/')   AS slashed
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("Acme Corp", "PUNE", "bcd", 9, "2023/01/05")]

### NULL in string functions

`concat` follows the usual NULL rule: if any argument is `NULL`, the result is `NULL`. `concat_ws` skips `NULL`
arguments, so it is the safer choice for building a label from optional parts such as a middle name.

In [ ]:
q = "SELECT concat('a', NULL, 'b') AS with_concat, concat_ws('-', 'a', NULL, 'b') AS with_concat_ws"
spark.sql(q).show()
assert spark.sql(q).collect() == [(None, "a-b")]

### Splitting and pattern matching

`split(email, '@')` returns an array such as `['ana', 'x.com']`. Array subscripts with `[ ]` start at **0**, so
`[1]` is the second element, the domain. This differs from `substring` and `element_at`, which count from 1.

`regexp_extract(email, '^([a-z]+)', 1)` returns capture group 1: the run of letters at the start, the username.
`RLIKE` tests a regular expression; `'[.]org$'` means "ends in `.org`". Writing the dot inside brackets matches
a literal dot without needing a backslash.

In [ ]:
q = """
SELECT id,
       email,
       split(email, '@')[1]                       AS domain,
       regexp_extract(email, '^([a-z]+)', 1)     AS username,
       email RLIKE '[.]org$'                     AS is_org
FROM contacts
ORDER BY id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [
    (1, "ana@x.com", "x.com", "ana", False),
    (2, "ben@y.org", "y.org", "ben", True),
    (3, "cy@x.com", "x.com", "cy", False),
]

## 4. Dates and times

`DATE` holds a day. `TIMESTAMP` holds a day and a time. Date arithmetic is done with functions or with
`INTERVAL` literals:

| Need | Use |
| --- | --- |
| add or subtract days | `date_add(d, n)`, `date_sub(d, n)` |
| difference in days | `datediff(later, earlier)` |
| add months | `add_months(d, n)` |
| parts of a date | `year(d)`, `month(d)`, `extract(MONTH FROM d)` |
| start of a period | `date_trunc('month', ts)`, `trunc(d, 'MM')` |
| parse text | `to_date('07/05/2023', 'dd/MM/yyyy')` |
| format as text | `date_format(d, 'yyyy-MM')` |

Use a fixed reference date in examples, not `current_date()`. A query that depends on today's date returns a
different result every day, which makes it hard to check.

### Where you'll use it

- **Month-end and week-start reporting:** `date_trunc` buckets events into the period a finance team reports on.
- **SLA calculations:** `datediff` between order and delivery timestamps.

Worked by hand for a reference date of 2023-04-01: order 1 was placed on 2023-01-05. From 5 January to 5 April
is 31 + 28 + 31 = 90 days, and 1 April is 4 days earlier, so 86 days. Order 14, placed on 2023-03-25, is 7 days
before 1 April.

In [ ]:
q = """
SELECT order_id,
       order_date,
       datediff(DATE '2023-04-01', order_date) AS days_open,
       month(order_date)                       AS month_number
FROM orders
WHERE order_id IN (1, 14)
ORDER BY order_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, date(2023, 1, 5), 86, 1), (14, date(2023, 3, 25), 7, 3)]

Three results below are worth checking by hand:

- Adding one month to 31 January gives **28 February**. February has no 31st, so the result is clamped to the
  last day of the month. `INTERVAL 1 MONTH` does the same.
- `date_trunc('month', ...)` returns a `TIMESTAMP` at midnight on the first of the month.
- `to_date('07/05/2023', 'dd/MM/yyyy')` is **7 May**, not 5 July. The pattern decides which number is the day.

In [ ]:
from datetime import datetime

q = """
SELECT add_months(DATE '2023-01-31', 1)            AS one_month_later,
       DATE '2023-01-31' + INTERVAL 1 MONTH        AS same_with_interval,
       TIMESTAMP '2023-01-01 10:00:00' - INTERVAL 30 MINUTES AS half_hour_earlier,
       date_trunc('month', TIMESTAMP '2023-05-07 10:20:00') AS month_start,
       to_date('07/05/2023', 'dd/MM/yyyy')        AS parsed
"""
spark.sql(q).show(truncate=False)
assert spark.sql(q).collect() == [(
    date(2023, 2, 28),
    date(2023, 2, 28),
    datetime(2023, 1, 1, 9, 30),
    datetime(2023, 5, 1, 0, 0),
    date(2023, 5, 7),
)]

### Bucketing by month

To count orders per month, turn each date into a month label and group by it. `date_format(order_date,
'yyyy-MM')` gives a sortable text label. January has orders 1, 4, 7 and 11; February has 2, 5, 8, 9 and 13;
March has 3, 6, 10, 12 and 14.

Pattern letters are case-sensitive: `MM` is the month and `mm` is minutes. `date_format(ts, 'yyyy-mm')` on
10:20 returns `'2023-20'`, a silent bug.

In [ ]:
q = """
SELECT date_format(order_date, 'yyyy-MM') AS order_month, COUNT(*) AS orders
FROM orders
GROUP BY date_format(order_date, 'yyyy-MM')
ORDER BY order_month
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("2023-01", 4), ("2023-02", 5), ("2023-03", 5)]

q = "SELECT date_format(TIMESTAMP '2023-05-07 10:20:00', 'yyyy-mm') AS minutes_not_month"
assert spark.sql(q).collect() == [("2023-20",)]

A `TIMESTAMP` in Spark is an instant in time. It is stored in UTC and displayed in the session time zone
(`spark.sql.session.timeZone`). Two sessions with different time zones show the same stored value as different
clock times. `TIMESTAMP_NTZ` ("no time zone") stores the clock reading as written, with no conversion.

## 5. Arrays, structs and maps

An **array** is an ordered list of values of one type. A **struct** is a record with named fields, read with a
dot: `s.city`. A **map** is a set of key-value pairs, read with a key: `m['b']`. These types let one row hold
nested data, such as a list of items in an order.

### Where you'll use it

- **Event payloads:** clickstream events with a list of product IDs per session, exploded for one row per product.
- **Nested analytics:** BigQuery and Spark store nested JSON as structs and arrays, so you query them with these functions.

`element_at` counts from 1, so `element_at(array('x', 'y'), 2)` is `'y'`. Reading past the end of an array with
`[ ]` or `element_at` raises an error in ANSI mode; `try_element_at` returns `NULL`. A missing map key returns
`NULL`.

In [ ]:
q = """
SELECT size(array(1, 2, 3))            AS element_count,
       array_contains(array(1, 2), 2)  AS has_two,
       element_at(array('x', 'y'), 2)  AS second_element,
       try_element_at(array('x', 'y'), 5) AS past_the_end,
       named_struct('city', 'Pune', 'zip', '411001').city AS city_field,
       map('a', 1, 'b', 2)['b']        AS map_value,
       map('a', 1, 'b', 2)['z']        AS missing_key
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(3, True, "y", None, "Pune", 2, None)]

### explode and collect_list

`explode` turns an array into one row per element. The other columns of the row are repeated on each new row.
Two sessions with arrays of 2 and 1 products become 3 rows. A row whose array is empty or `NULL` produces no rows
at all; `explode_outer` keeps it, with `NULL` as the element.

In [ ]:
sessions = """
SELECT * FROM VALUES
  ('s1', array('p1', 'p2')),
  ('s2', array('p3')),
  ('s3', CAST(array() AS ARRAY<STRING>))
AS sessions(session_id, products)
"""

q = f"SELECT session_id, explode(products) AS product FROM ({sessions}) ORDER BY session_id, product"
spark.sql(q).show()
assert spark.sql(q).collect() == [("s1", "p1"), ("s1", "p2"), ("s2", "p3")]

q = f"SELECT session_id, explode_outer(products) AS product FROM ({sessions}) ORDER BY session_id, product"
assert spark.sql(q).collect() == [("s1", "p1"), ("s1", "p2"), ("s2", "p3"), ("s3", None)]

`collect_list` does the reverse: it is an aggregate that gathers a group's values into an array. The order of
the elements is not guaranteed, so wrap it in `array_sort` when the order matters. `collect_set` also removes
duplicates.

In [ ]:
q = """
SELECT customer_id, array_sort(collect_list(order_id)) AS order_ids
FROM orders
GROUP BY customer_id
ORDER BY customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [
    (1, [1, 2, 3]), (2, [4, 5, 6]), (3, [7, 8]), (4, [9, 10]), (5, [11, 12]), (6, [13, 14]),
]

## 6. JSON

JSON often arrives as text. Spark can read a field with `get_json_object`, which returns a string, or parse the
whole document with `from_json` against a schema, which returns a typed struct. `to_json` goes the other way.

A JSON **path** such as `'$.city'` names a field: `$` is the whole document, `.city` is a field and `[0]` is
the first element of an array.

### Where you'll use it

- **API landing tables:** a raw JSON column in the warehouse, parsed in a downstream model.
- **Semi-structured logs:** pulling one field out of a JSON event payload when the schema changes often.

In [ ]:
q = """
SELECT id,
       get_json_object(address_json, '$.city')   AS city,
       from_json(address_json, 'city STRING, zip STRING').zip AS zip
FROM shipments
ORDER BY id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, "Pune", "411001"), (2, "Delhi", "110001")]

q = "SELECT to_json(named_struct('city', 'Pune')) AS as_json"
spark.sql(q).show(truncate=False)
assert spark.sql(q).collect() == [('{"city":"Pune"}',)]

### Strings versus typed values

`get_json_object` always returns a `STRING`, even for a number: from `{"a": 1}` it returns `'1'`. A path that
does not exist returns `NULL`, and so does text that is not valid JSON; neither raises an error, so check for
`NULL`s after parsing. `from_json` with the schema `'a INT'` returns the number as an `INT`. The schema must
match the JSON as written: `1` is a JSON number, but `"1"` is a JSON string.

In [ ]:
q = """
SELECT get_json_object('{"a": 1}', '$.a')          AS as_string,
       typeof(get_json_object('{"a": 1}', '$.a'))  AS string_type,
       get_json_object('{"a": 1}', '$.b')          AS missing_path,
       get_json_object('not json', '$.a')          AS invalid_json,
       from_json('{"a": 1}', 'a INT').a + 1        AS typed_plus_one
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("1", "string", None, None, 2)]

### Nested JSON: parse, then explode

An order payload often holds an array of items. `from_json` with an `ARRAY<STRUCT<...>>` schema turns it into
an array of structs; `explode` gives one row per item; ordinary SQL does the rest. The payload below has two
items with quantities 2 and 1, so the total quantity is 3.

In [ ]:
q = """
WITH payload AS (
  SELECT '{"order_id": 7, "items": [{"sku": "A", "qty": 2}, {"sku": "B", "qty": 1}]}' AS body
),
parsed AS (
  SELECT from_json(body, 'order_id INT, items ARRAY<STRUCT<sku: STRING, qty: INT>>') AS doc
  FROM payload
),
items AS (
  SELECT doc.order_id AS order_id, explode(doc.items) AS item FROM parsed
)
SELECT order_id, item.sku AS sku, item.qty AS qty FROM items ORDER BY sku
"""
spark.sql(q).show()
rows = spark.sql(q).collect()
assert rows == [(7, "A", 2), (7, "B", 1)]
assert sum(row.qty for row in rows) == 3

## Interview notes

- **Money.** Use `DECIMAL`, not `DOUBLE`. Binary floating point cannot represent 0.1 exactly, so sums drift.
  The next cell adds the two values in both types.
- **Integer division.** `/` returns a double in Spark, but some engines (PostgreSQL, SQL Server) truncate
  `10 / 4` to `2`. Know which one your engine does before you compute an average or a ratio.
- **ANSI mode.** Bad casts, overflow and division by zero raise errors. The `try_` functions (`try_cast`,
  `try_add`, `try_divide`, `try_element_at`) return `NULL` instead.
- **Implicit conversion.** When the types differ, the engine converts one side. Joining an `INT` column to a
  `STRING` column can silently stop matching rows or slow the query. Keep join keys the same type.
- **Indexing.** Array `[ ]` is 0-based; `substring` and `element_at` are 1-based.
- **Date patterns.** `MM` is month, `mm` is minutes; `dd/MM` and `MM/dd` read the same text as different dates.

In [ ]:
from decimal import Decimal

q = """
SELECT CAST(0.1 AS DOUBLE) + CAST(0.2 AS DOUBLE) AS as_double,
       0.1 + 0.2                                  AS as_decimal
"""
spark.sql(q).show()
as_double, as_decimal = spark.sql(q).first()
assert as_double == 0.30000000000000004
assert as_decimal == Decimal("0.3")

## Practice questions

1. How many orders were placed in March 2023? Count them.
2. How many contacts does each email domain have? Show the domain and the count, sorted by domain.
3. Parse each shipment's address with `from_json`, returning `id`, `city` and `zip`, with `zip` as an `INT`.
   Sort by `id`.
4. What does `0.1 + 0.2` return in Spark when both values are literals, and why is that different from doing the same sum with `DOUBLE`?
5. For each customer who has ordered, how many days passed between signing up and their first order? Sort by
   `customer_id`.
6. For each customer who has ordered, list the distinct statuses they have used as a sorted array.
7. The text values `'12'`, `'0042'`, `'n/a'` and `' 7 '` arrive in a column. How many convert to `INT`, and
   what is the sum of those that do?

## Answers

### Question 1

Extract the month and year from `order_date` and filter on both. March has orders 3, 6, 10, 12 and 14.

In [ ]:
q = "SELECT COUNT(*) AS march_orders FROM orders WHERE month(order_date) = 3 AND year(order_date) = 2023"
spark.sql(q).show()
assert spark.sql(q).collect() == [(5,)]

### Question 2

Split each email on `@`, take element `[1]` (0-based) as the domain, and group by it.

In [ ]:
q = """
SELECT split(email, '@')[1] AS domain, COUNT(*) AS contacts
FROM contacts
GROUP BY split(email, '@')[1]
ORDER BY domain
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("x.com", 2), ("y.org", 1)]

### Question 3

In the JSON text the zip is quoted, `"zip": "411001"`, so it is a JSON string. Parse it with the type it really
has, `zip STRING`, then `CAST` it to `INT` in SQL. Reading fields from the parsed struct in a CTE keeps the query
short.

A schema of `zip INT` looks simpler but is wrong: the JSON parser does not convert a quoted string to a number,
and it returns `NULL` for the field without raising an error. The second check proves it.

In [ ]:
q = """
WITH parsed AS (
  SELECT id, from_json(address_json, 'city STRING, zip STRING') AS address FROM shipments
)
SELECT id, address.city AS city, CAST(address.zip AS INT) AS zip
FROM parsed
ORDER BY id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, "Pune", 411001), (2, "Delhi", 110001)]

wrong = "SELECT from_json(address_json, 'city STRING, zip INT').zip AS zip FROM shipments ORDER BY id"
assert spark.sql(wrong).collect() == [(None,), (None,)]

### Question 4

Both literals are `DECIMAL`, so the sum is exact and equals `0.3`. Python's `float` and Spark's `DOUBLE` are
binary floating point, where `0.1 + 0.2` is `0.30000000000000004`. The next cell checks both results.

In [ ]:
q = "SELECT 0.1 + 0.2 AS decimal_total, CAST(0.1 AS DOUBLE) + CAST(0.2 AS DOUBLE) AS double_total"
spark.sql(q).show()
decimal_total, double_total = spark.sql(q).first()
assert decimal_total == Decimal("0.3")
assert double_total != 0.3

### Question 5

Find each customer's first order with `MIN(order_date)`, join to `customers`, and use `datediff`. Customer 1
signed up on 2022-01-05 and first ordered on 2023-01-05: exactly one non-leap year, 365 days. Customer 2: from
2022-02-10 to 2023-02-10 is 365 days, and 2023-01-20 is 21 days earlier, so 344.

In [ ]:
q = """
WITH first_orders AS (
  SELECT customer_id, MIN(order_date) AS first_order
  FROM orders
  GROUP BY customer_id
)
SELECT c.customer_id, datediff(f.first_order, c.signup_date) AS days_to_first_order
FROM customers c
JOIN first_orders f ON f.customer_id = c.customer_id
ORDER BY c.customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 365), (2, 344), (3, 316), (4, 300), (5, 222), (6, 233)]

### Question 6

`collect_set` gathers the distinct values of each group into an array; `array_sort` makes the order fixed.

In [ ]:
q = """
SELECT customer_id, array_sort(collect_set(status)) AS statuses
FROM orders
GROUP BY customer_id
ORDER BY customer_id
"""
spark.sql(q).show(truncate=False)
assert spark.sql(q).collect() == [
    (1, ["completed"]),
    (2, ["cancelled", "completed"]),
    (3, ["completed"]),
    (4, ["completed", "pending"]),
    (5, ["completed"]),
    (6, ["cancelled", "completed"]),
]

### Question 7

`try_cast` returns `NULL` for text that is not a whole number, and `COUNT` and `SUM` skip `NULL`s. `'0042'`
converts to 42 and `' 7 '` to 7, because leading zeros and surrounding spaces are allowed. Only `'n/a'` fails.
Three values convert, summing to 12 + 42 + 7 = 61.

In [ ]:
q = """
SELECT COUNT(try_cast(raw AS INT)) AS converted, SUM(try_cast(raw AS INT)) AS total
FROM VALUES ('12'), ('0042'), ('n/a'), (' 7 ') AS t(raw)
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(3, 61)]